# 🚀 YOLOv7 Custom Training in Colab
This notebook sets up and runs YOLOv7 training with AMP and WandB support on Google Colab.
**Prepared on:** 2025-04-19 17:21:51

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


## Upload your prepared `yolov7_colab_portable.zip`

In [ ]:
# Clean up any old yolov7 directories or ZIPs
!rm -rf /content/yolov7 /content/yolov7_colab_portable.zip /content/yolov7_colab_portable\ \(1\).zip


In [ ]:
from google.colab import files
uploaded = files.upload()  # Upload yolov7_colab_portable.zip


Saving yolov7_colab_portable.zip to yolov7_colab_portable.zip


In [ ]:
!cp "/content/drive/MyDrive/yolov7_colab_portable.zip" /content/

In [ ]:
!unzip -o yolov7_colab_portable.zip -d /content/
!ls /content/yolov7



Archive:  yolov7_colab_portable.zip
   creating: /content/yolov7/
   creating: /content/yolov7/paper/
  inflating: /content/yolov7/paper/yolov7.pdf  
  inflating: /content/yolov7/LICENSE.md  
   creating: /content/yolov7/tools/
  inflating: /content/yolov7/tools/compare_YOLOv7_vs_YOLOv5s6.ipynb  
  inflating: /content/yolov7/tools/compare_YOLOv7_vs_YOLOv5m6.ipynb  
  inflating: /content/yolov7/tools/visualization.ipynb  
  inflating: /content/yolov7/tools/compare_YOLOv7_vs_YOLOv5m6_half.ipynb  
  inflating: /content/yolov7/tools/instance.ipynb  
  inflating: /content/yolov7/tools/YOLOv7trt.ipynb  
  inflating: /content/yolov7/tools/compare_YOLOv7e6_vs_YOLOv5x6_half.ipynb  
  inflating: /content/yolov7/tools/YOLOv7CoreML.ipynb  
  inflating: /content/yolov7/tools/YOLOv7-Dynamic-Batch-TENSORRT.ipynb  
  inflating: /content/yolov7/tools/keypoint.ipynb  
  inflating: /content/yolov7/tools/YOLOv7-Dynamic-Batch-ONNXRUNTIME.ipynb  
  inflating: /content/yolov7/tools/compare_YOLOv7e6_vs_YOLOv5

In [ ]:
!cp "/content/drive/MyDrive/RAGChat_yolo_dataset.zip" /content/

In [ ]:
!unzip -o /content/RAGChat_yolo_dataset.zip -d /content/yolov7/


Streaming output truncated to the last 5000 lines.
 extracting: /content/yolov7/dataset/labels/train/0367__fliph__rot-45_jpg.rf.81ad191ba46ed80986cae9bd2b31fedd.txt  
 extracting: /content/yolov7/dataset/labels/train/0009__rot25__trans20_10_jpg.rf.13f0850d38d4a37fe67d9cd9eb3fb358.txt  
 extracting: /content/yolov7/dataset/labels/train/800px-QC_08_jpg.rf.a76a6dd1a099c325f8680e3c883cb7fb.txt  
 extracting: /content/yolov7/dataset/labels/train/HelloIMG1648042365053_jpeg.rf.85ca399edf9f109ad9834842027ef4f5.txt  
 extracting: /content/yolov7/dataset/labels/train/XIAEYBNQZYQ9_jpg.rf.85744c881b8a635f1ca57d0c20cd3a91.txt  
 extracting: /content/yolov7/dataset/labels/train/0162__rot25__trans20_10_jpg.rf.ddb7d8bc23d89a1e3f9d149473742c5b.txt  
 extracting: /content/yolov7/dataset/labels/train/V2-frame-127_jpg.rf.92fa787118c7d791f0dcb1c64b915c22.txt  
 extracting: /content/yolov7/dataset/labels/train/6c5e635b2d9ed352_jpg.rf.67cf08e24021a7decd106b1e68059d2c.txt  
 extracting: /content/yolov7/datase

In [ ]:
!ls /content/yolov7/dataset
!ls /content/yolov7/

annotations  dataset.yaml  images  labels
cfg	   export.py	LICENSE.md   requirements_modified.txt	tools
data	   figure	models	     requirements.txt		train_aux.py
dataset    hubconf.py	paper	     scripts			train.py
deploy	   inference	__pycache__  setup.py			utils
detect.py  __init__.py	README.md    test.py			yolov7.egg-info


In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


CUDA available: True
GPU: NVIDIA A100-SXM4-40GB


## Install Dependencies

In [ ]:
%cd /content/yolov7
!pip install -r requirements.txt


/content/yolov7


In [ ]:

!cp -r /content/yolov7 /content/drive/MyDrive/yolov7_colab_backup

In [ ]:
!cp /content/drive/MyDrive/yolov7_colab_backup/train.py /content/yolov7/train.py
!cp /content/drive/MyDrive/yolov7_colab_backup/utils/loss.py /content/yolov7/utils/loss.py
!cp /content/drive/MyDrive/yolov7_colab_backup/utils/general.py /content/yolov7/utils/general.py

In [ ]:
!find /content/yolov7 -type f -name "*.py" -exec sed -i 's/from yolov7\.utils/from utils/g' {} +
!find /content/yolov7 -type f -name "*.py" -exec sed -i 's/import yolov7\.utils/import utils/g' {} +


In [ ]:
import os

print("Train dir exists:", os.path.exists("/content/yolov7/dataset/images/train"))
print("Val dir exists:", os.path.exists("/content/yolov7/dataset/images/val"))

print("Number of train images:", len(os.listdir("/content/yolov7/dataset/images/train")) if os.path.exists("/content/yolov7/dataset/images/train") else 0)
print("Number of val images:", len(os.listdir("/content/yolov7/dataset/images/val")) if os.path.exists("/content/yolov7/dataset/images/val") else 0)


Train dir exists: True
Val dir exists: True
Number of train images: 8720
Number of val images: 2180


## Run YOLOv7 Training

In [ ]:
!rm -rf /content/yolov7/dataset/labels/*.cache


In [ ]:
!ls /content/yolov7/dataset/labels/train | wc -l
!ls /content/yolov7/dataset/labels/val | wc -l


8720
2180


In [ ]:
!python -u train.py \
  --img 640 \
  --batch 16 \
  --epochs 50 \
  --data dataset/dataset.yaml \
  --cfg cfg/training/yolov7-tiny-custom.yaml \
  --weights '' \
  --name yolov7-tiny-colab \
  --hyp data/hyp.scratch.p5.yaml
